In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
file_path = "/Users/a/Programming/Langchain-Project/external-data/UAS Capstone Project.pdf"
loader = PyPDFLoader(file_path)

In [8]:
docs = loader.load_and_split()

In [9]:
print(docs[10])

page_content='BAB  I  
PENDAHULUAN
   Pada  era  sekarang,  kebutuhan  informasi  yang  cepat  dan  tepat  menjadi  salah  satu  
kebutuhan
 
penting
 
bagi
 
setiap
 
orang.
 
Begitupun
 
dalam
 
bidang
 
akademik,
 
pelayanan
 
informasi
 
menjadi
 
aspek
 
penting
 
bagi
 
stakeholder
 
untuk
 
mendapatkan
 
informasi
 
[1].
 
Dalam
 
praktiknya,
 
stakeholder
 
menginginkan
 
layanan
 
informasi
 
yang
 
dinamis
 
atau
 
bisa
 
disesuaikan
 
berdasarkan
 
kebutuhan
 
(
customised)
 
[2].
 
Namun
 
layanan
 
yang
 
sekarang
 
masih
 
memanfaatkan
 
tenaga
 
manusia
 
(staff)
 
masih
 
memiliki
 
banyak
 
sekali
 
kekurangan,
 
terutama
 
dalam
 
aspek
 
kecepatan,
 
responsivitas
 
dan
 
prosedur
 
dalam
 
memberikan
 
informasi
 
[1]
  1.1  Latar  Belakang    Pada  era  digital  saat  ini,  akses  terhadap  informasi  yang  cepat  dan  presisi  telah  menjadi  
kebutuhan
 
fundamental,
 
tidak
 
terkecuali
 
dalam
 
ekosistem
 
akademik.
 
Pemangku
 
kepentingan
 
(
stakeholder
)
 

In [11]:
import pprint

pprint.pp(docs[0].metadata)

{'producer': 'Skia/PDF m145 Google Docs Renderer',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'UAS Capstone Project - 20251',
 'source': '/Users/a/Programming/Langchain-Project/external-data/UAS Capstone '
           'Project.pdf',
 'total_pages': 39,
 'page': 0,
 'page_label': '1'}


In [12]:
pprint.pp(docs[15].metadata)

{'producer': 'Skia/PDF m145 Google Docs Renderer',
 'creator': 'PyPDF',
 'creationdate': '',
 'title': 'UAS Capstone Project - 20251',
 'source': '/Users/a/Programming/Langchain-Project/external-data/UAS Capstone '
           'Project.pdf',
 'total_pages': 39,
 'page': 15,
 'page_label': '16'}


bagian ini adalah splitting data pdf document menjadi potongan-potongnan atau chunking

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
docs[15].page_content[:500]

'2.3.5  Tahap  Pengujian  dan  Evaluasi  \nTahap  akhir  ini  bertujuan  untuk  memvalidasi  apakah  solusi  yang  dibangun  telah  \nmenjawab\n \nrumusan\n \nmasalah.\n \nA.  Uji  Fungsional:  Memastikan  seluruh  komponen  sistem  berjalan  tanpa  error  \ndi\n \nlingkungan\n \nlokal.\n B.  Evaluasi  Akurasi  RAG:  Mengukur  kualitas  jawaban  sistem  menggunakan  \nmetrik\n \nevaluasi\n \n(seperti\n \nrelevansi\n \njawaban\n \ndan\n \ntingkat\n \nhalusinasi)\n \nuntuk\n \nmemastikan\n \ninformasi\n \nyang\n \ndiberikan\n \nakurat\n \nd'

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    keep_separator=False,
)

In [16]:
all_splits = text_splitter.split_documents(docs)

In [17]:
print(f"split blog post into {len(all_splits)} sub-documents.")

split blog post into 144 sub-documents.


In [18]:
all_splits[3]

Document(metadata={'producer': 'Skia/PDF m145 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'UAS Capstone Project - 20251', 'source': '/Users/a/Programming/Langchain-Project/external-data/UAS Capstone Project.pdf', 'total_pages': 39, 'page': 3, 'page_label': '4', 'start_index': 0}, page_content='KATA  PENGANTAR   Puji  syukur  penulis  panjatkan  kehadirat  Allah  SWT,  karena  atas  berkat  dan  rahmat-  \nNya,\n \npenulis\n \ndapat\n \nmenyelesaikan\n \nLaporan\n \nAkhir\n \nCapstone\n \nProject\n \nperiode\n \ntahun\n \nakademik\n \n2025/2026\n \nini.\n \nPenulis\n \nmenyadari\n \nbahwa,\n \ntanpa\n \nbantuan\n \ndan\n \nbimbingan\n \ndari\n \nberbagai\n \npihak,\n \ndalam\n \npelaksanaan\n \nkegiatan\n \nCapstone\n \nProject\n \nini,\n \ntidaklah\n \nmudah\n \nbagi\n \npenulis\n \nuntuk\n \nmenyelesaikan\n \nLaporan\n \nCapstone\n \nProject\n \nini.\n \nOleh')

embedding model tetap dengan indo bert dan vecto store tetap menggunakan chromdb

In [24]:
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOllama
from langchain_chroma import Chroma
from transformers import BertTokenizer, AutoModel
from typing import List
from langchain_core.embeddings import Embeddings
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
import torch

In [3]:
llm_model = ChatOllama(
    model='mistral-openorca:7b-q4_0',
    temperature=0,
    streaming=True,
)

In [4]:
class IndoBertEmbeddings(Embeddings):
    def __init__(self, model_name="indobenchmark/indobert-base-p1"):
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()


    def _generate_embedding(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

        with torch.no_grad():
            outputs = self.model(**inputs)

        token_embeddings = outputs.last_hidden_state

        sentence_embeddings = token_embeddings.mean(dim=1)

        return sentence_embeddings.squeeze().tolist()
    

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._generate_embedding(text) for text in texts]
    

    # pencarian query
    def embed_query(self, text: str) -> List[float]:
        return self._generate_embedding(text)

embeddings_model = IndoBertEmbeddings()


In [6]:
vector_store = Chroma(
    collection_name="pdf-data",
    embedding_function=embeddings_model,
    persist_directory="./pdf-database"
)


In [19]:
vector_store.add_documents(all_splits)

['90b253b5-c245-4c25-833d-8216ef30bdf4',
 'f833a5ee-bd9b-466d-8381-16789a7142f9',
 'dbbd9f31-cf2f-43dd-8faf-21a5906011a8',
 'dc29f3d1-a461-41aa-8bf6-a6f1af4896a6',
 '1930d082-b893-49b4-8ef0-2e4a48b8f328',
 '8991fd88-0b44-4544-9b33-00a414fcb6a4',
 'd90181ca-8f3b-4116-89a3-009cff954b1a',
 'a67d4c7c-111b-4eeb-a833-ac167669a5f9',
 '4809f416-54a2-411b-a84f-13f0e61b9744',
 '578ec0af-76a3-4355-b529-5dfa54216f03',
 'e69498f8-fb8f-4278-a315-c54fbd5e92d4',
 'd5d2b6de-d0d7-4be5-850b-edc89cdd2773',
 '3ba9798e-136e-491c-9da4-ca2514a2ea5f',
 'cf56dbe3-d516-4bc5-b3cf-1fecdc2c3d9a',
 '48c93576-7c78-4608-8d9e-31664857db38',
 '9fd629b9-e2c4-41da-9290-051a03f88d0a',
 'afccd729-9d20-4a67-bc30-0f6abd55a44a',
 '5bcdcaa8-6408-47b2-b015-93f2612e1646',
 '164bc374-da60-469e-b5ca-1c0b923db60d',
 '0948c0d0-0fd8-4098-8bf0-753854330b8e',
 '716be357-f7a9-42f5-b933-e1f0903376a4',
 '58e9b542-1728-45fa-958e-f334a02f849b',
 '341bbb14-2bf7-4d31-adda-419a43ccdce0',
 '373d4160-3a01-45fa-8ad5-32980b93e6ca',
 'fd8eaaf4-590c-

In [20]:
retriever = vector_store.as_retriever()

In [25]:
prompt = ChatPromptTemplate.from_template(
    """Kamu adalah asisten cerdas. Jawablah pertanyaan pengguna berdasarkan konteks berikut ini saja.
    
    KONTEKS:
    {context}
    
    INSTRUKSI TAMBAHAN:
    1. Jawablah pertanyaan berdasarkan KONTEKS di atas.
    2. Kamu WAJIB menjawab dalam Bahasa Indonesia, terlepas dari bahasa apa pertanyaan itu diberikan.
    3. Jika jawaban tidak ada di dalam konteks, katakan "Maaf, informasi tidak ditemukan dalam dokumen."
    
    PERTANYAAN: {question}
    JAWABAN:"""
)

In [22]:
setup_and_retrieval = RunnableParallel(
    {
        "context":retriever,
        "question":RunnablePassthrough()
    }
)

In [54]:
chain = setup_and_retrieval | prompt | llm_model | StrOutputParser()

In [66]:
chain.invoke("apa metode yang digunakan untuk pencarian dokumen")

' Metode yang digunakan untuk pencarian dokumen adalah retriever (pencarian).'

In [37]:
query = "rumusan masalah pada penelitian"

In [38]:
result = vector_store.similarity_search(query=query, k=2)

In [39]:
for doc in result:
    print(doc.page_content)

penelitian  ini  untuk  dikembangkan.  Membuat  sistem  yang  bisa  membantu  stakeholder  
untuk
 
mendapatkan
 
informasi
 
yang
 
dinamis.
 
Lebih
 
lanjut
 
lagi
 
membuat
 
sistem
 
yang
 
bisa
 
menjadi
 
penjelas
 
atau
 
explanatory
 
dari
 
informasi
 
statis
 
yang
 
didapatkan
 
pada
 
dashboard
 
akademik.
 
   2.3  Tahapan  Pelaksanaan  Proyek  
Untuk  memastikan  tercapainya  tujuan  penelitian  yang  telah  didefinisikan  pada  bagian  
rumusan
 
masalah
 
dan
 
tujuan
rumusan
 
masalah
 
dan
 
tujuan
 
penelitian,
 
pelaksanaan
 
proyek
 
ini
 
disusun
 
dalam
 
lima
 
tahapan
 
sistematis.
 
Alur
 
kerja
 
ini
 
dirancang
 
untuk
 
menjembatani
 
kesenjangan
 
antara
 
informasi
 
statis
 
yang
 
tersedia
 
saat
 
ini
 
dengan
 
kebutuhan
 
informasi
 
dinamis
 
stakeholder
.
 
2.3.1  Tahap  Studi  Pendahuluan  dan  Analisis  Masalah  
Tahap  ini  merupakan  fondasi  proyek  yang  bertujuan  untuk  membedah  akar  
permasalahan
 
di
 
lingkungan
 
studi


menggunakan api key dari gemini

In [41]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    
)

In [52]:
chain_gemini = setup_and_retrieval | prompt | model | StrOutputParser()

In [65]:
chain_gemini.invoke("apa metode yang digunakan untuk pencarian dokumen")

'Metode yang digunakan untuk pencarian dokumen adalah dengan menggunakan komponen **Retriever**. Berdasarkan konteks, komponen ini merupakan bagian dari arsitektur sistem **Retrieval-Augmented Generation (RAG)** yang dirancang untuk memberikan informasi dinamis.'